# Практична робота: параметричне моделювання L-подібного кронштейна

**Мета:** описати технічну деталь системою параметрів і залежностей, щоб модель автоматично перебудовувалася після зміни розмірів.

Кронштейн складається з двох взаємно перпендикулярних полиць однакової товщини `T`. Параметри: ширина `W`, висота `H`, глибина `D`, товщина `T`, діаметр отворів `d`, відступ отворів від країв `E`.

Система координат: `X` — глибина, `Y` — висота, `Z` — ширина. Бічний профіль екструдується вздовж осі Z.

**Важливо:** отвори в цій навчальній візуалізації позначені контурами на поверхнях. Це параметрична геометрична модель, а не булеве вирізання отворів у твердотільному ядрі.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import ipywidgets as widgets
from IPython.display import display

plt.rcParams['figure.figsize'] = (14, 9)


## 1. Геометрична модель

Профіль у площині XY задається точками `(0,0), (D,0), (D,T), (T,T), (T,H), (0,H)` і екструдується на ширину `W`.

Два отвори на кожній полиці розміщуються симетрично:
\[
z_1=E,\qquad z_2=W-E.
\]
Отже, їхня середина завжди дорівнює `W/2`.

In [ ]:
def validate(W, H, D, T, d, E):
    if min(W, H, D, T, d, E) <= 0:
        raise ValueError("Усі параметри мають бути додатними.")
    if T >= min(H, D):
        raise ValueError("Товщина T має бути меншою за H і D.")
    if 2*E >= W or E <= d/2:
        raise ValueError("Перевірте відступ E та діаметр отвору d.")
    if E + d/2 >= min(H-T, D-T):
        raise ValueError("Для отворів недостатньо місця. Зменште d або E.")


def draw_prism(ax, profile, W):
    a = [(x,y,0) for x,y in profile]
    b = [(x,y,W) for x,y in profile]
    faces = [a, b]
    for i in range(len(profile)):
        j = (i+1) % len(profile)
        faces.append([a[i], a[j], b[j], b[i]])
    mesh = Poly3DCollection(faces, facecolors="#a9c4d6",
                            edgecolors="#263746", linewidths=0.8, alpha=0.9)
    ax.add_collection3d(mesh)


def draw_bracket(W=100, H=80, D=70, T=6, d=10, E=15):
    validate(W,H,D,T,d,E)
    r = d/2
    zholes = [E, W-E]
    xhole = D*0.68
    yhole = H*0.62
    profile = [(0,0),(D,0),(D,T),(T,T),(T,H),(0,H)]

    fig = plt.figure(figsize=(15,10))
    ax = fig.add_subplot(221, projection="3d")
    top = fig.add_subplot(222)
    front = fig.add_subplot(223)
    side = fig.add_subplot(224)

    draw_prism(ax, profile, W)
    ang = np.linspace(0,2*np.pi,120)
    for z in zholes:
        # Контур отвору на верхній поверхні горизонтальної полиці (площина XZ)
        ax.plot(xhole+r*np.cos(ang), np.full_like(ang,T),
                z+r*np.sin(ang), color="darkred", lw=2)
        # Контур отвору на зовнішній поверхні вертикальної полиці (площина YZ)
        ax.plot(np.full_like(ang,T), yhole+r*np.cos(ang),
                z+r*np.sin(ang), color="darkred", lw=2)

    ax.set(xlim=(0,D*1.08), ylim=(0,H*1.08), zlim=(0,W),
           xlabel="X — глибина", ylabel="Y — висота", zlabel="Z — ширина",
           title="Просторова модель")
    ax.set_box_aspect((D,H,W))
    ax.view_init(elev=24, azim=-55)

    # Вид зверху: горизонтальна полиця, координати X-Z
    top.fill([0,D,D,0],[0,0,W,W], color="#a9c4d6", alpha=.8)
    for z in zholes:
        top.add_patch(plt.Circle((xhole,z),r,fill=False,color="darkred",lw=2))
    top.set(xlim=(-5,D+5),ylim=(-5,W+5),xlabel="X — глибина",
            ylabel="Z — ширина",title="Вигляд зверху")
    top.set_aspect("equal")

    # Вид спереду: вертикальна полиця, координати Z-Y
    front.fill([0,W,W,0],[0,0,H,H], color="#a9c4d6", alpha=.8)
    for z in zholes:
        front.add_patch(plt.Circle((z,yhole),r,fill=False,color="darkred",lw=2))
    front.set(xlim=(-5,W+5),ylim=(-5,H+5),xlabel="Z — ширина",
              ylabel="Y — висота",title="Вигляд спереду")
    front.set_aspect("equal")

    # Вид збоку: L-подібний профіль
    closed = profile + [profile[0]]
    side.fill([p[0] for p in closed],[p[1] for p in closed],
              color="#a9c4d6",alpha=.8)
    side.plot([p[0] for p in closed],[p[1] for p in closed],color="#263746")
    side.set(xlim=(-5,D+5),ylim=(-5,H+5),xlabel="X — глибина",
             ylabel="Y — висота",title="Вигляд збоку")
    side.set_aspect("equal")

    fig.suptitle(f"L-кронштейн: W={W} мм, H={H} мм, D={D} мм, "
                 f"T={T} мм, d={d} мм, E={E} мм",fontsize=14)
    plt.tight_layout()
    plt.show()

draw_bracket()


## 2. Інтерактивне керування

Змінюйте параметри — просторове зображення та проєкції перебудовуються.

In [ ]:
W = widgets.IntSlider(value=100,min=60,max=180,step=5,description="W, мм")
H = widgets.IntSlider(value=80,min=50,max=150,step=5,description="H, мм")
D = widgets.IntSlider(value=70,min=50,max=140,step=5,description="D, мм")
T = widgets.IntSlider(value=6,min=3,max=12,step=1,description="T, мм")
d = widgets.IntSlider(value=10,min=6,max=18,step=1,description="d, мм")
E = widgets.IntSlider(value=15,min=10,max=30,step=1,description="E, мм")

controls = widgets.VBox([widgets.HBox([W,H]), widgets.HBox([D,T]),
                          widgets.HBox([d,E])])
output = widgets.interactive_output(draw_bracket,
    {"W":W,"H":H,"D":D,"T":T,"d":d,"E":E})
display(controls, output)


## 3. Аналіз параметричних залежностей

- Координати отворів по ширині обчислюються як `E` та `W-E`.
- Середина між отворами завжди дорівнює `W/2`, тому симетрія зберігається при зміні ширини.
- Зміна `H` не змінює `T`; зміна `D` не змінює `W`.
- Координата отворів уздовж полиці задається відносно її габариту.

## 4. Самостійна робота

1. Збільште `W` зі 100 до 140 мм. Обчисліть відстань між отворами.
2. Змініть `E` з 15 до 20 мм. Поясніть, чому обидва отвори переміщуються симетрично.
3. Змініть товщину `T`. Які габарити залишаються незмінними?
4. Запропонуйте параметр, який варто додати до моделі: наприклад, кількість отворів або відстань між рядами.
5. Сформулюйте три геометричні залежності, які має зберігати кронштейн.

## 5. Контрольні запитання

1. Чим параметрична модель відрізняється від звичайного креслення?
2. Які параметри є незалежними, а які — похідними?
3. Як математично забезпечується симетрія отворів?
4. Чому зміна одного параметра не повинна вимагати ручного перемальовування деталі?

**Очікуваний результат:** студент може не лише продемонструвати модель, а й пояснити правила, за якими вона формується.